# **Feature Engineering**

## Objectives

* Split the data into train and test sets
* Build a reproducible preprocessing pipeline that applies the validated cleaning and feature transformation steps identified during exploratory data cleaning, together with the encoding shared across all planned machine learning tasks.
* Fit the pipeline on train data and transform both train and test sets 


## Inputs

* outputs/datasets/cleaned/HotelBookingsValid.csv

## Outputs

* Feature engineering pipeline saved as outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl
* X_train, X_test, y_train and y_test to outputs/ml_pipeline/preprocessing as .csv

## Additional Comments

* The preprocessing pipeline created in this notebook is designed to be universal to classification, regression and clustering tasks in order to facilitate future implementation of customer segmentation and cancel window prediction. 
* Model specific preprocessing tasks are carried out in the modelling stages.
* Clustering and regression are descoped for this iteration of the project, but the framework to allow for them in the future remains.


---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir

# Load Data

* Load the validated dataset created in the [cleaning notebook](/jupyter_notebooks/04_cleaning.ipynb)

In [ ]:
import pandas as pd

df = pd.read_csv("outputs/datasets/cleaned/HotelBookingsValid.csv")
df.head(3)

---

## Split train and test set

* The validated dataset is separated into features (X) and target (y), before being split into training and testing subsets using an 80:20 ratio. Stratified sampling is used to preserve the proportion of cancelled and non-cancelled bookings in both datasets, ensuring that model evaluation is representative of the original class distribution.

In [ ]:
from sklearn.model_selection import train_test_split

df_data = df.drop("is_canceled", axis=1)
target = df["is_canceled"]

print(f"df_data shape: {df_data.shape}, target shape: {target.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    df_data, target, test_size=0.2, random_state=4, stratify=target
)

display_shapes = pd.Series({"X_train": X_train.shape,
              "X_test": X_test.shape,
              "y_train": y_train.shape,
              "y_test": y_test.shape}, name="split_shapes")
display_shapes

* The split produced 95,350 training observations and 23,838 testing observations. Separating the test set before preprocessing prevents information leakage, allowing all transformations to be fitted exclusively on the training data and then applied consistently to unseen data.

---

## Pipeline

**Planned Preprocessing Actions**

| Feature | Meaning | Data type | Preprocessing actions | 
| --- | --- | --- | --- |
| hotel | booking location | nominal | one-hot |
| is_canceled | booking was cancelled | binary | none |
| lead_time | days before arrival booking made | numeric | winsorize |
| arrival_date_year | year of arrival | numeric | drop |
| arrival_date_month | month of arrival | nominal | defer |
| arrival_date_week_number | week of the year of arrival | numeric | none |
| arrival_date_day_of_month | day in the month of arrival | numeric | none |
| stays_in_weekend_nights | weekend nights stayed | numeric | winsorize |
| stays_in_week_nights | week nights stayed | numeric | winsorize |
| adults | number of adults on the booking | numeric | none |
| children | number of children | numeric | none |
| babies | number of babies | numeric | none |
| meal | meal plan booked | nominal | replace "Undefined" with "SC", one-hot |
| country | country of origin | nominal | impute missing data |
| market_segment | demographic information | nominal | one-hot |
| distribution_channel | demographic information | nominal | one-hot |
| is_repeated_guest | if the guest has booked before | binary | none |
| previous_cancellations | how many times the guest has cancelled before | numeric | none |
| previous_bookings_not_canceled | how many times the guest has completed a booking | numeric | none |
| reserved_room_type | Room code booked | nominal | one-hot |
| deposit_type | Booking security policy | nominal | one-hot |
| agent | ID code of booking agent | nominal | impute missing data |
| company | ID code of company the guest is travelling for | nominal | drop |
| days_in_waiting_list | How long the booking waited for confirmation | numeric | none |
| customer_type | demographic information | nominal | one-hot |
| adr | cost per night of the booking | numeric | winsorize |
| required_car_parking_spaces | car parking spaces needed | numeric | none |
| total_of_special_requests | special requests made | numeric | none |

**Cleaning steps**

The preprocessing pipeline reproduces the cleaning decisions validated in the [cleaning](/jupyter_notebooks/04_cleaning.ipynb) notebook. Each transformation is applied sequentially to ensure consistent handling of new data.
1. Drop `company` and `arrival_date_year`
2. Replace "Undefined" with "SC" in `meal`
3. Impute missing values
4. Outlier handling

**Feature Engineering Steps**

The preprocessing pipeline introduces One-Hot encoding for the low-cardinality categorical variables

In [ ]:
print("X_train shape ", X_train.shape)

### 1. Drop Columns

* The `company` feature contains a very high proportion of missing values and was identified during data cleaning as unsuitable for modelling. `arrival_date_year` contains only three years and is excluded to improve generalisability, avoiding a model that learns year-specific booking behaviour.
* Both are removed

In [ ]:
from feature_engine.selection import DropFeatures

X_train_copy = X_train.copy()
drop_transformer = DropFeatures(features_to_drop=["company", "arrival_date_year"])
pipeline_step1 = drop_transformer.fit_transform(X_train_copy)
pipeline_step1.shape


* The new shape (95350, 25) illustrates that the 2 features are successfully removed

### 2. Replace "Undefined" with "SC"

* The meal variable contains a small number (~1%) of "Undefined" values. During data cleaning these were determined to represent the same booking behaviour as "SC", allowing them to be merged into a single category before encoding.

In [ ]:
from sklearn.preprocessing import FunctionTransformer
from src.custom_transformers import undefined_meal

replace_transformer = FunctionTransformer(undefined_meal)
pipeline_step2 = replace_transformer.fit_transform(pipeline_step1)
pipeline_step2["meal"].value_counts()

* The replacement removes the redundant category while preserving all observations.

### 3. Impute Missing Values

* Missing values in agent indicate that no booking agent was used. These values are therefore imputed with 0, creating an explicit category rather than estimating a value.
* Confirm that the value 0 is not already used as a valid agent identifier before applying the imputation.

In [ ]:
pipeline_step2["agent"].describe()

* Check missing levels for `agent`

In [ ]:
pipeline_step2["agent"].isnull().sum()

* Run pipeline imputation step

In [ ]:
from feature_engine.imputation import ArbitraryNumberImputer, CategoricalImputer

agent_imputer = ArbitraryNumberImputer(arbitrary_number=0, variables="agent")
pipeline_step3 = agent_imputer.fit_transform(pipeline_step2)
print("Missing values: ", pipeline_step3["agent"].isnull().sum())
pipeline_step3["agent"].describe()


* Verification shows that the new minimum value is 0 and there are no more missing values

* Missing values in country are comparatively rare and are imputed using the most frequent category. This approach preserves all observations while introducing minimal distortion to the distribution.
* Run the imputer pipeline step

In [ ]:
country_imputer = CategoricalImputer(imputation_method="frequent", variables="country")
pipeline_step4 = country_imputer.fit_transform(pipeline_step3)
print("Missing values: ", pipeline_step4["country"].isnull().sum())
pipeline_step4["country"].value_counts()

* Verification shows that there are now no missing values in the `country` variable

### 4. Outlier Handling 

In [ ]:
pipeline_step4.describe()

* Boxplots are used to verify the variables identified during exploratory analysis as containing substantial right-tailed outliers.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math

outlier_cols = ["lead_time", "adr", "stays_in_weekend_nights", "stays_in_week_nights"]

def plot_outliers(cols):
    ncols = 2
    nrows = math.ceil(len(cols) / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(12, 8))
    fig.suptitle("Outlier Distributions")
    axs = axs.flatten()

    for i, col in enumerate(cols):
        sns.boxplot(data=pipeline_step4,
                     x=col,
                     ax=axs[i])   

    plt.tight_layout()

plot_outliers(outlier_cols)

* The distributions confirm the presence of extreme values in the selected numerical variables.

* Rather than removing observations, winsorisation caps extreme values using the IQR method. This reduces the influence of unusually large values while preserving the size of the training dataset. This approach is more robust than removing observations entirely, as similarly extreme values are likely to occur in future booking data.
* Run the Winsorizer pipeline step

In [ ]:
from feature_engine.outliers import Winsorizer

winsorizer = Winsorizer(capping_method="iqr", tail="right", fold=1.5, variables=outlier_cols)
pipeline_step5 = winsorizer.fit_transform(pipeline_step4)
pipeline_step5.describe()

* Re-plot the variables to assess the new distribution

In [ ]:
def plot_outliers(cols):
    ncols = 2
    nrows = math.ceil(len(cols) / ncols)
    fig, axs = plt.subplots(nrows, ncols, figsize=(12, 8))
    fig.suptitle("Transformed Distributions")
    axs = axs.flatten()

    for i, col in enumerate(cols):
        sns.boxplot(data=pipeline_step5,
                     x=col,
                     ax=axs[i])   

    plt.tight_layout()

plot_outliers(outlier_cols)

* The capped distributions show that extreme observations have been reduced while the underlying distributions remain intact.

### One-hot encode categorical variables

* Machine learning algorithms generally require numerical input. The low-cardinality, nominal categorical variables are therefore transformed using one-hot encoding. The final category is dropped to avoid perfect multicollinearity.

In [ ]:
categorical_cols = ["hotel", "meal", "market_segment", "distribution_channel", "reserved_room_type", "deposit_type", "customer_type"]
for col in categorical_cols:
    print(f"{col} has {df[col].nunique()} categories")

* Run the One-Hot encoding pipeline step

In [ ]:
from feature_engine.encoding import OneHotEncoder

encoder = OneHotEncoder(variables=categorical_cols, drop_last=True)
pipeline_step6 = encoder.fit_transform(pipeline_step5)
pipeline_step6.head(3)

* Nominal variables requiring the same treatment across all machine learning tasks have now been one-hot encoded.

* Assemble the preprocessing pipeline

In [ ]:
from sklearn.pipeline import Pipeline

def preprocessing_pipeline():

    pipeline_base = Pipeline([
        ("DropFeatures", DropFeatures(features_to_drop=["company", "arrival_date_year"])),
        ("FunctionTransformer", FunctionTransformer(undefined_meal)),
        ("ArbitraryNumberImputer", ArbitraryNumberImputer(arbitrary_number=0, variables="agent")),
        ("CategoricalImputer", CategoricalImputer(imputation_method="frequent", variables="country")),
        ("Winsorizer", Winsorizer(capping_method="iqr", tail="right", fold=1.5, variables=outlier_cols)),
        ("OneHotEncoder", OneHotEncoder(variables=categorical_cols, drop_last=True))
    ])

    return pipeline_base

* Fit the pipeline to the train set and transform both sets

In [ ]:
pipeline_preprocessing = preprocessing_pipeline()

X_train_processed = pipeline_preprocessing.fit_transform(X_train)

X_test_processed = pipeline_preprocessing.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

In [ ]:
X_train_processed.head()

* The transformed training data confirms that the preprocessing pipeline executes successfully. All defined preprocessing steps have been applied sequentially while preserving deferred variables for model-specific processing in later notebooks.

---

## Save Files

In [ ]:
import os
try:
  os.makedirs(name='outputs/ml_pipeline/preprocessing')
except Exception as e:
  print(e)


* Save the un-transformed train and test sets for use in the modelling notebooks

**Train Set**

In [ ]:
X_train.to_csv("outputs/ml_pipeline/preprocessing/X_train.csv", index=False)
y_train.to_csv("outputs/ml_pipeline/preprocessing/y_train.csv", index=False)

**Test Set**

In [ ]:
X_test.to_csv("outputs/ml_pipeline/preprocessing/X_test.csv", index=False)
y_test.to_csv("outputs/ml_pipeline/preprocessing/y_test.csv", index=False)

**Pipeline**

In [ ]:
pipeline_preprocessing = preprocessing_pipeline()
pipeline_preprocessing

In [ ]:
import joblib

pipeline_preprocessing = preprocessing_pipeline()

joblib.dump(value=pipeline_preprocessing,
            filename="outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl")

---

## Conclusion

The preprocessing pipeline has been successfully constructed and saved alongside the train and test datasets. All validated cleaning operations, missing value handling, outlier treatment and categorical encoding have been encapsulated within a reusable Scikit-learn pipeline. This ensures consistent preprocessing across future modelling tasks while preventing data leakage by fitting transformations only on the training data.

## Next Steps

Model-specific transformations are intentionally deferred to the modelling notebooks, allowing the same preprocessing framework to support future classification, regression and clustering workflows while preventing data leakage through fitting exclusively on the training data.

* Proceed to classification modelling, applying appropriate tree-based model preprocessing
* Assess the best algorithm
* Tune the model for performance
